In [1]:
import pandas as pd
import numpy as np
from scipy.stats import zscore
from sklearn.preprocessing import StandardScaler
pd.set_option('display.max_columns', 100)

In [2]:
# Assuming get_player_data and get_sc_data are defined functions
player_raw = pd.read_csv('2024_player_stats_current.csv')
player_raw.rename(columns={'round': 'round_number'}, inplace=True)
draft_raw = pd.read_csv('2024_draft_leagues_1283_recap_combined.csv')
df_coach_list = pd.read_csv('../../inputs/coach_list.csv', index_col=0)
draft_raw = draft_raw.merge(df_coach_list, left_on='user_team_id', right_on='coach_team_id', how='left')
player_raw = player_raw.merge(draft_raw, on='player_id', how='left')

# Calculate player stats
rnds = player_raw['played'].max() # THIS MIGHT NOT BE THE BEST WAY TO DEFINE NUMBER OF GAMES PLAYED, CHECK IT'S CORRECT
player_raw['avg_adj'] = round(((player_raw['avg'] * player_raw['played']) + ((rnds - player_raw['played']) * np.where(player_raw['avg'] < 75, player_raw['avg'], 75))) / rnds, 1)
# player_raw['round_pos_rank_for_all_players'] = player_raw.groupby(['round_number', 'pos_1'])['points'].rank(ascending=False)
# player_raw['round_pos_rank_for_drafted_players'] = player_raw[player_raw['user_team_id'].notnull()].groupby(['round_number', 'pos_1'])['points'].rank(ascending=False)
# player_raw.loc[player_raw['player_id'] == 273]
player_raw


,feed_id,player_id,first_name,last_name,team_abbrev,pos_1,pos_2,round_number,points,played,avg,avg3,avg5,price,id,league_id,user_team_id,round,pick,position,autopicked,time,coach_id,coach_team_id,coach_first_name,coach_team_name,avg_adj
0,1012807,1,Sam,Berry,ADE,MID,NaN,1,80,1,80.0000,80.0000,80.00,226900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,75.2
1,1012807,1,Sam,Berry,ADE,MID,NaN,2,52,2,66.0000,66.0000,66.00,226900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,66.0
2,1012807,1,Sam,Berry,ADE,MID,NaN,4,33,3,55.0000,55.0000,55.00,243100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,55.0
3,1012807,1,Sam,Berry,ADE,MID,NaN,5,56,4,55.2500,47.0000,55.25,244400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,55.2
4,1012807,1,Sam,Berry,ADE,MID,NaN,6,31,5,50.4000,40.0000,50.40,235500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9512,996731,99,Charlie,Curnow,CAR,FWD,NaN,18,89,17,85.6471,58.6667,74.80,394800,1705978.0,1283.0,718.0,8.0,59.0,FWD,0.0,8/03/2024 13:26,8258.0,718.0,Simon,Wilkie Wonka,82.9
9513,996731,99,Charlie,Curnow,CAR,FWD,NaN,19,106,18,86.7778,76.0000,80.00,393700,1705978.0,1283.0,718.0,8.0,59.0,FWD,0.0,8/03/2024 13:26,8258.0,718.0,Simon,Wilkie Wonka,84.2
9514,996731,99,Charlie,Curnow,CAR,FWD,NaN,20,119,19,88.4737,104.6670,80.20,431500,1705978.0,1283.0,718.0,8.0,59.0,FWD,0.0,8/03/2024 13:26,8258.0,718.0,Simon,Wilkie Wonka,86.1
9515,996731,99,Charlie,Curnow,CAR,FWD,NaN,21,33,20,85.7000,86.0000,76.00,433900,1705978.0,1283.0,718.0,8.0,59.0,FWD,0.0,8/03/2024 13:26,8258.0,718.0,Simon,Wilkie Wonka,84.3


In [3]:
# Standardize averages
pos_list = pd.DataFrame({
    'p': ['DEF', 'MID', 'RUC', 'FWD'],
    'n': [5, 7, 2, 5]
})
pos_list['sum_n'] = pos_list['n'].sum()
pos_list['weight'] = round(pos_list['n'] / pos_list['sum_n'], 3)
pos_list

,p,n,sum_n,weight
0,DEF,5,19,0.263
1,MID,7,19,0.368
2,RUC,2,19,0.105
3,FWD,5,19,0.263


In [4]:
scr_data = pd.DataFrame()
overall_smy = pd.DataFrame()

for i, row in pos_list.iterrows():
    p = row['p']
    n = row['n']
    
    pos_data = player_raw[player_raw['pos_1'].str.contains(p) | player_raw['pos_2'].str.contains(p)].copy()
    pos_data.loc[:, 'scr_pos'] = p
    
    pos_data = pos_data.pivot_table(index=['feed_id','player_id', 'scr_pos'], values=['points'], aggfunc=['count','mean']).reset_index()
    pos_data.columns = pos_data.columns.get_level_values(0)
    pos_data.rename(columns={'mean': 'avg'}, inplace=True)
    
    pos_smy = pos_data[['player_id', 'avg']].sort_values(by='avg', ascending=False).head(n * 8)
    
    mean = pos_smy['avg'].mean()
    sd = pos_smy['avg'].std()
    
    pos_data.loc[:, 'pos_scr'] = round((pos_data['avg'] - mean) / sd, 3)
    
    pos_data.loc[:, 'mean'] = mean
    pos_data.loc[:, 'sd'] = sd
    scr_data = pd.concat([scr_data, pos_data])
    overall_smy = pd.concat([overall_smy, pos_smy])

scr_data = scr_data.sort_values(by='pos_scr', ascending=False).drop_duplicates(subset='feed_id')
scr_data.to_csv('scr_data.csv')
scr_data

,feed_id,player_id,scr_pos,count,avg,pos_scr,mean,sd
171,1009260,273,FWD,22,119.045455,3.216,85.034961,10.575400
55,298539,666,FWD,21,116.761905,3.000,85.034961,10.575400
1,261224,86,FWD,23,110.434783,2.402,85.034961,10.575400
52,297373,695,MID,23,126.391304,2.400,102.879972,9.797867
219,1023518,509,DEF,21,117.952381,2.182,96.985311,9.610428
...,...,...,...,...,...,...,...,...
229,1027872,618,DEF,5,20.400000,-7.969,96.985311,9.610428
268,1023787,698,FWD,1,0.000000,-8.041,85.034961,10.575400
172,1012817,201,DEF,1,15.000000,-8.531,96.985311,9.610428
223,1024096,680,DEF,2,14.000000,-8.635,96.985311,9.610428


In [10]:
mean = overall_smy['avg'].mean()
sd = overall_smy['avg'].std()

heat_map_data2 = player_raw.copy()
heat_map_data2['scr'] = round((heat_map_data2['avg'] - mean) / sd, 3)
heat_map_data2 = heat_map_data2.merge(scr_data[['player_id', 'scr_pos', 'count', 'pos_scr', 'mean', 'sd']], on='player_id', how='left')
heat_map_data2 = heat_map_data2.merge(pos_list[['p', 'weight']], left_on='scr_pos', right_on='p', how='left')
heat_map_data2['scr_wgt'] = heat_map_data2['pos_scr'] * heat_map_data2['weight'] + heat_map_data2['scr'] * (1 - heat_map_data2['weight'])
heat_map_data2['name'] = heat_map_data2['first_name'].str[0] + '.' + heat_map_data2['last_name']
heat_map_data2['round_rank_by_pos_all_players'] = heat_map_data2.groupby(['round_number', 'scr_pos'])['points'].rank(ascending=False)
heat_map_data2['round_rank_by_pos_drafted_players'] = heat_map_data2[heat_map_data2['user_team_id'].notnull()].groupby(['round_number', 'scr_pos'])['points'].rank(ascending=False)
heat_map_data2 = heat_map_data2[['player_id', 'feed_id', 'name', 'team_abbrev', 'played', 'points', 'round_number', 'avg', 'weight', 'scr', 'scr_wgt', 'scr_pos', 'avg_adj', 'round_rank_by_pos_all_players', 'round_rank_by_pos_drafted_players']]
heat_map_data2 = heat_map_data2.sort_values(by=['round_number', 'round_rank_by_pos_all_players'], ascending=True)
heat_map_data2

,player_id,feed_id,name,team_abbrev,played,points,round_number,avg,weight,scr,scr_wgt,scr_pos,avg_adj,round_rank_by_pos_all_players,round_rank_by_pos_drafted_players
5034,474,291902,J.Viney,MEL,1,138,0,138.0000,0.368,3.334,1.548096,MID,77.7,1.0,1.0
7571,662,293957,B.Grundy,SYD,1,139,0,139.0000,0.105,3.415,3.083305,RUC,77.8,1.0,1.0
7640,666,298539,I.Heeney,SYD,1,144,0,144.0000,0.263,3.819,3.603603,FWD,78.0,1.0,1.0
502,138,1023261,N.Daicos,COL,1,131,0,131.0000,0.263,2.768,2.592579,DEF,77.4,1.5,1.5
3343,348,1009253,L.Ash,GWS,1,131,0,131.0000,0.263,2.768,1.705480,DEF,77.4,1.5,1.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5337,498,1001017,N.Larkey,NTH,23,14,24,60.9565,0.263,-2.896,-2.733203,FWD,61.0,163.0,NaN
5243,486,1023475,B.Drury,NTH,6,12,24,38.8333,0.263,-4.685,-4.601892,FWD,38.8,164.0,NaN
3416,351,1014038,C.Brown,GWS,13,11,24,49.1538,0.263,-3.850,-3.729809,FWD,49.2,165.0,NaN
1159,182,996232,M.Guelfi,ESS,13,10,24,71.3846,0.263,-2.053,-1.852594,FWD,71.4,166.0,NaN


In [11]:
# Calculate Draft data
draft_data = draft_raw.copy()
draft_data = draft_data.sort_values(by=['pick'])
draft_data = draft_data.drop_duplicates()
draft_data = draft_data[['user_team_id', 'coach_first_name', 'round', 'pick', 'player_id']].merge(heat_map_data2, on='player_id', how='right')
draft_data = draft_data.sort_values(by=['round', 'pick', 'scr_wgt'], ascending=[True, True, False])
draft_data['roll_rank'] = draft_data.groupby('round')['scr_wgt'].transform(lambda x: x.rolling(8, min_periods=1).rank(ascending=False))
# draft_data.to_csv('kek.csv')
draft_data

,user_team_id,coach_first_name,round,pick,player_id,feed_id,name,team_abbrev,played,points,round_number,avg,weight,scr,scr_wgt,scr_pos,avg_adj,round_rank_by_pos_all_players,round_rank_by_pos_drafted_players,roll_rank
625,714.0,Mark,1.0,1.0,695,297373,M.Bontempelli,WBD,2,136,2,131.000,0.368,2.768,2.632576,MID,79.9,7.0,6.0,1.0
1348,714.0,Mark,1.0,1.0,695,297373,M.Bontempelli,WBD,4,141,4,128.250,0.368,2.546,2.492272,MID,84.3,4.0,4.0,2.0
5797,714.0,Mark,1.0,1.0,695,297373,M.Bontempelli,WBD,15,157,16,128.133,0.368,2.536,2.485952,MID,109.7,1.0,1.0,3.0
7864,714.0,Mark,1.0,1.0,695,297373,M.Bontempelli,WBD,20,182,21,127.450,0.368,2.481,2.451192,MID,120.6,1.0,1.0,4.0
8300,714.0,Mark,1.0,1.0,695,297373,M.Bontempelli,WBD,21,119,22,127.048,0.368,2.449,2.430968,MID,122.5,6.5,5.5,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2480,NaN,NaN,NaN,NaN,787,1031819,T.Sellers,NTH,2,6,6,4.500,0.263,-7.461,-7.501502,FWD,4.5,152.0,NaN,NaN
821,NaN,NaN,NaN,NaN,328,1021103,M.Knevitt,GEE,1,0,2,0.000,0.368,-7.825,-7.504472,MID,0.0,70.0,NaN,NaN
2113,NaN,NaN,NaN,NaN,787,1031819,T.Sellers,NTH,1,3,5,3.000,0.263,-7.582,-7.590679,FWD,3.0,161.0,NaN,NaN
545,NaN,NaN,NaN,NaN,284,996464,D.Macpherson,GCS,2,1,1,8.000,0.263,-7.178,-7.725303,DEF,8.0,133.0,NaN,NaN


In [17]:
pick_order = draft_data[['pick', 'coach_first_name']].drop_duplicates().sort_values(by='pick').dropna().head(8)
pick_order_dict = pick_order.set_index('pick')['coach_first_name'].to_dict()
draft_summary = draft_data.pivot_table(index=['round', 'pick', 'coach_first_name', 'player_id', 'feed_id', 'name', 'team_abbrev','scr_pos'], values=['points', 'round_rank_by_pos_drafted_players', 'round_rank_by_pos_all_players'], aggfunc='mean').reset_index()
ordered_columns = ['round'] + [pick_order_dict[pick] for pick in sorted(pick_order_dict.keys())]


draft_summary.rename(columns={'points': 'avg'}, inplace=True)
draft_summary['scr'] = round((draft_summary['avg'] - mean) / sd, 3)
draft_summary = draft_summary.merge(scr_data[['player_id','pos_scr']], on='player_id', how='left')
draft_summary = draft_summary.merge(pos_list, left_on='scr_pos', right_on='p', how='left')
draft_summary['scr_wgt'] = round(draft_summary['pos_scr'] * draft_summary['weight'] + draft_summary['scr'] * (1 - draft_summary['weight']), 3)
draft_summary

,round,pick,coach_first_name,player_id,feed_id,name,team_abbrev,scr_pos,avg,round_rank_by_pos_all_players,round_rank_by_pos_drafted_players,scr,pos_scr,p,n,sum_n,weight,scr_wgt
0,1.0,1.0,Mark,695,297373,M.Bontempelli,WBD,MID,126.391304,11.282609,9.804348,2.395,2.400,MID,7,19,0.368,2.397
1,1.0,2.0,Luke,138,1023261,N.Daicos,COL,DEF,117.173913,21.630435,12.478261,1.650,2.101,DEF,5,19,0.263,1.769
2,1.0,3.0,Simon,706,1004592,T.English,WBD,RUC,108.000000,7.318182,6.318182,0.908,0.401,RUC,2,19,0.105,0.855
3,1.0,4.0,Lester,459,298210,C.Petracca,MEL,MID,103.230769,23.653846,19.230769,0.523,0.036,MID,7,19,0.368,0.344
4,1.0,5.0,Paul,443,290528,M.Gawn,MEL,RUC,124.142857,4.785714,4.214286,2.214,2.087,RUC,2,19,0.105,2.201
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
179,23.0,180.0,Lester,503,1024272,C.McKercher,NTH,DEF,88.500000,45.750000,23.125000,-0.669,-0.883,DEF,5,19,0.263,-0.725
180,23.0,181.0,Paul,263,1008454,C.Budarick,GCS,DEF,63.375000,77.375000,36.937500,-2.700,-3.497,DEF,5,19,0.263,-2.910
181,23.0,182.0,James,386,280109,C.Ward,GWS,MID,77.388889,41.472222,29.944444,-1.567,-2.602,MID,7,19,0.368,-1.948
182,23.0,183.0,Anthony,721,1002404,A.Naughton,WBD,FWD,70.578947,71.710526,26.315789,-2.118,-1.367,FWD,5,19,0.263,-1.920


In [18]:
import numpy as np
from scipy.stats import rankdata
############################################################################
# Function to calculate the rank based on the next 7 picks
def get_nearest_7_rank(row, df):
    current_pick = row['pick']
    if row['round'] == df['round'].max():
        selected_picks = df[(df['round'] == df['round'].max())]
    else:
        selected_picks = df[(df['pick'] > current_pick) & (df['pick'] <= current_pick + 7)]
    if not selected_picks.empty:
        combined_list = list(selected_picks['scr_wgt']) + [row['scr_wgt']]
        ranks = rankdata(-np.array(combined_list), method='min')
        value_rank = ranks[-1]
        return value_rank # return rankdata(-next_8_picks['scr_wgt'], method='min')[0]
    return np.nan

# Apply the function to create the 'nearest_8_rank' column
# draft_summary = draft_summary.head(30)
draft_summary['nearest_7_rank'] = draft_summary.apply(lambda row: get_nearest_7_rank(row, draft_summary), axis=1)
draft_summary.to_csv('kek2.csv')
draft_summary

,round,pick,coach_first_name,player_id,feed_id,name,team_abbrev,scr_pos,avg,round_rank_by_pos_all_players,round_rank_by_pos_drafted_players,scr,pos_scr,p,n,sum_n,weight,scr_wgt,nearest_7_rank
0,1.0,1.0,Mark,695,297373,M.Bontempelli,WBD,MID,126.391304,11.282609,9.804348,2.395,2.400,MID,7,19,0.368,2.397,1
1,1.0,2.0,Luke,138,1023261,N.Daicos,COL,DEF,117.173913,21.630435,12.478261,1.650,2.101,DEF,5,19,0.263,1.769,2
2,1.0,3.0,Simon,706,1004592,T.English,WBD,RUC,108.000000,7.318182,6.318182,0.908,0.401,RUC,2,19,0.105,0.855,4
3,1.0,4.0,Lester,459,298210,C.Petracca,MEL,MID,103.230769,23.653846,19.230769,0.523,0.036,MID,7,19,0.368,0.344,6
4,1.0,5.0,Paul,443,290528,M.Gawn,MEL,RUC,124.142857,4.785714,4.214286,2.214,2.087,RUC,2,19,0.105,2.201,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
179,23.0,180.0,Lester,503,1024272,C.McKercher,NTH,DEF,88.500000,45.750000,23.125000,-0.669,-0.883,DEF,5,19,0.263,-0.725,3
180,23.0,181.0,Paul,263,1008454,C.Budarick,GCS,DEF,63.375000,77.375000,36.937500,-2.700,-3.497,DEF,5,19,0.263,-2.910,7
181,23.0,182.0,James,386,280109,C.Ward,GWS,MID,77.388889,41.472222,29.944444,-1.567,-2.602,MID,7,19,0.368,-1.948,5
182,23.0,183.0,Anthony,721,1002404,A.Naughton,WBD,FWD,70.578947,71.710526,26.315789,-2.118,-1.367,FWD,5,19,0.263,-1.920,4


In [ ]:
draft_table = draft_data.pivot_table(index='round', columns='coach_first_name', values='name', aggfunc='first').reset_index()
draft_table = draft_table[ordered_columns]
draft_table = draft_table.merge(draft_table_color, on='round', suffixes=('', '_color'))

# Visualization (using plotly or seaborn for heatmap)
import plotly.express as px

fig = px.imshow(draft_table.set_index('round').T, color_continuous_scale='RdYlGn', aspect='auto')
fig.show()

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import colors

# Define the color map
brks = [1, 3, 5, 7]
clrs = colors.ListedColormap(['#d73027', '#fc8d59', '#fee08b', '#d9ef8b', '#1a9850'])

# Function to apply background color
def color_background(val):
    if pd.isna(val):
        return ''
    val = float(val)  # Ensure the value is numeric
    for i, brk in enumerate(brks):
        if val <= brk:
            return f'background-color: {clrs.colors[i]}'
    return f'background-color: {clrs.colors[-1]}'

# Convert relevant columns to numeric
for col in draft_table.columns[1:]:
    draft_table[col] = pd.to_numeric(draft_table[col], errors='coerce')

# Apply the background color to the DataFrame
styled_table = draft_table.style.applymap(color_background, subset=pd.IndexSlice[:, draft_table.columns[1:]])

# Display the styled DataFrame
styled_table

In [ ]:
# Summary data
draft_data_smy = draft_data[draft_data['coach_first_name'].notna()].copy()
draft_data_smy['grp'] = pd.cut(draft_data_smy['roll_rank'], bins=[-np.inf, 1, 3, 5, 7, np.inf], labels=['1', '2-3', '4-5', '6-7', '8']).astype(str)
draft_data_smy = draft_data_smy.groupby(['coach_first_name', 'grp']).size().reset_index(name='n')
draft_data_smy['smy'] = draft_data_smy['grp'].map({'1': 1, '2-3': 2.5, '4-5': 4.5, '6-7': 6.5, '8': 8}) * draft_data_smy['n']
draft_data_smy

In [ ]:
draft_data_smy1 = draft_data_smy.groupby('coach_first_name')['smy'].sum().div(23).round(1).reset_index(name='Total')
draft_data_smy2 = draft_data_smy.pivot_table(index='grp', columns='coach_first_name', values='n', fill_value=0).reset_index()
draft_data_smy3 = pd.concat([draft_data_smy2, draft_data_smy1.T.reset_index().rename(columns={0: 'Avg'})])
draft_data_smy3